In [6]:
!pip install pmdarima

In [7]:
import warnings
import logging

import pandas as pd
import numpy as np

from math import sqrt
from typing import TypedDict, List, Dict, Optional

from sklearn.metrics import (mean_absolute_error,mean_squared_error)

from sklearn.model_selection import (ParameterGrid,TimeSeriesSplit)

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

from prophet import Prophet
from pmdarima import auto_arima
from langgraph.graph import (StateGraph,END)

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich import box

In [8]:
# =========================================================
# CONFIGURATION
# =========================================================

DATASET_PATH = "Final_Dataset.csv"
TARGET_COLUMN = "quantity_sold"
FORECAST_DAYS = 7

# =========================================================
# LOGGING
# =========================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
warnings.filterwarnings("ignore")
console = Console()

In [9]:
# =========================================================
# LOAD DATASET
# =========================================================

df = pd.read_csv(DATASET_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")
df.set_index("date", inplace=True)
df = df.asfreq("D")

# =========================================================
# FEATURE ENGINEERING
# =========================================================

df["lag1"] = df[TARGET_COLUMN].shift(1)
df["lag7"] = df[TARGET_COLUMN].shift(7)

df = df.dropna()

# =========================================================
# TRAIN TEST SPLIT
# =========================================================

split_index = int(len(df) * 0.8)
train = df.iloc[:split_index]
test = df.iloc[split_index:]
y_train = train[TARGET_COLUMN]
y_test = test[TARGET_COLUMN]

In [10]:
# =========================================================
# METRICS
# =========================================================

def evaluate(actual, pred):
    actual = np.array(actual)
    pred = np.array(pred)
    mae = mean_absolute_error(actual, pred)
    rmse = sqrt(mean_squared_error(actual, pred))
    mape = np.mean(
        np.abs((actual - pred) / (actual + 1e-10))
    ) * 100
    return {
        "MAE": round(float(mae), 3),
        "RMSE": round(float(rmse), 3),
        "MAPE": round(float(mape), 3)
    }


In [11]:
# =========================================================
# STATE
# =========================================================

class ForecastState(TypedDict):
    models: List[str]
    results: List[Dict]
    best_model: str
    tuning_method: str
    best_params: Dict
    forecast_df: pd.DataFrame
    error: Optional[str]

In [12]:
# =========================================================
# MODEL TRAINERS
# =========================================================

def run_arima():
    model = ARIMA(y_train, order=(1, 1, 1))
    fit = model.fit()
    pred = fit.forecast(len(y_test))
    metrics = evaluate(y_test, pred)
    return metrics

def run_sarimax():
    model = SARIMAX(y_train, order=(1, 1, 1))
    fit = model.fit()
    pred = fit.forecast(len(y_test))
    metrics = evaluate(y_test, pred)
    return metrics

def run_prophet():
    prophet_df = df.reset_index()[["date", TARGET_COLUMN]]
    prophet_df.columns = ["ds", "y"]
    train_p = prophet_df.iloc[:split_index]
    test_p = prophet_df.iloc[split_index:]
    model = Prophet()
    model.fit(train_p)
    future = model.make_future_dataframe(
        periods=len(test_p)
    )

    forecast = model.predict(future)
    pred = forecast["yhat"].iloc[-len(test_p):]
    metrics = evaluate(test_p["y"], pred)
    return metrics

In [13]:
# =========================================================
# AGENT 1 → PLANNER
# =========================================================

def planner_agent(state: ForecastState):
    console.print(
        "\n[bold cyan]Planner Agent Running[/bold cyan]"
    )
    models = [
        "ARIMA",
        "SARIMAX",
        "Prophet"
    ]
    if len(df) > 500:
        models.append("LSTM")

    if len(df) > 2000:
        models.append("TCN")
    console.print(
        f"Selected Models → {models}"
    )
    return {
        "models": models
    }

In [14]:
# =========================================================
# AGENT 2 → EXECUTOR
# =========================================================

def executor_agent(state: ForecastState):

    console.print(
        "\n[bold yellow]Executor Agent Running[/bold yellow]"
    )
    results = []

    for model_name in state["models"]:
        try:
            if model_name == "ARIMA":
                metrics = run_arima()
            elif model_name == "SARIMAX":
                metrics = run_sarimax()
            elif model_name == "Prophet":
                metrics = run_prophet()
            else:
                continue
            result = {
                "Model": model_name,
                **metrics
            }
            results.append(result)
            console.print(
                f"[green]{model_name} Completed[/green]"
            )
        except Exception as e:
            logger.exception(e)
    return {
        "results": results
    }

In [15]:
# =========================================================
# AGENT 3 → CHECKER
# =========================================================

def checker_agent(state: ForecastState):
    console.print(
        "\n[bold green]Checker Agent[/bold green]"
    )
    results_df = pd.DataFrame(state["results"])
    table = Table(
        title="Model Evaluation Results",
        box=box.DOUBLE_EDGE
    )
    for col in results_df.columns:
        table.add_column(col)
    for _, row in results_df.iterrows():
        table.add_row(
            *[str(i) for i in row]
        )
    console.print(table)
    return {}

In [16]:
# =========================================================
# AGENT 4 → DECISION
# =========================================================

def decision_agent(state: ForecastState):
    console.print(
        "\n[bold magenta]Decision Agent[/bold magenta]"
    )
    results_df = pd.DataFrame(state["results"])
    best_row = results_df.sort_values(
        "RMSE"
    ).iloc[0]
    best_model = best_row["Model"]
    console.print(
        f"Best Model → {best_model}"
    )
    return {
        "best_model": best_model
    }


In [17]:
# =========================================================
# AGENT 5 → TUNING
# =========================================================

def tuning_agent(state: ForecastState):

    console.print(
        "\n[bold blue]Tuning Agent[/bold blue]"
    )
    param_grid = {
        "p": [0, 1, 2],
        "d": [0, 1],
        "q": [0, 1, 2]
    }
    best_score = np.inf
    best_params = None
    for params in ParameterGrid(param_grid):
        try:
            model = ARIMA(
                y_train,
                order=(
                    params["p"],
                    params["d"],
                    params["q"]
                )
            ).fit()
            pred = model.forecast(len(y_test))
            rmse = evaluate(
                y_test,
                pred
            )["RMSE"]
            if rmse < best_score:
                best_score = rmse
                best_params = params
        except:
            pass

    auto_model = auto_arima(
        y_train,
        seasonal=False
    )

    auto_pred = auto_model.predict(
        n_periods=len(y_test)
    )

    auto_rmse = evaluate(
        y_test,
        auto_pred
    )["RMSE"]

    scores = {
        "GridSearch": best_score,
        "AutoARIMA": auto_rmse
    }

    best_method = min(
        scores,
        key=scores.get
    )

    console.print(
        f"Best Tuning Method → {best_method}"
    )

    console.print(
        f"Best Params → {best_params}"
    )

    return {
        "tuning_method": best_method,
        "best_params": best_params
    }



In [18]:
# =========================================================
# AGENT 6 → FORECASTING
# =========================================================

def forecasting_agent(state: ForecastState):
    console.print(
        "\n[bold red]Forecasting Agent[/bold red]"
    )
    best_model = state["best_model"]
    y_full = df[TARGET_COLUMN]
    if best_model == "ARIMA":
        model = ARIMA(
            y_full,
            order=(1, 1, 1)
        )
        fit = model.fit()
        forecast = fit.forecast(
            steps=FORECAST_DAYS
        )
    elif best_model == "SARIMAX":
        model = SARIMAX(
            y_full,
            order=(1, 1, 1)
        )
        fit = model.fit()
        forecast = fit.forecast(
            steps=FORECAST_DAYS
        )
    elif best_model == "Prophet":
        prophet_df = df.reset_index()[
            ["date", TARGET_COLUMN]
        ]
        prophet_df.columns = ["ds", "y"]
        model = Prophet()
        model.fit(prophet_df)
        future = model.make_future_dataframe(
            periods=FORECAST_DAYS
        )
        forecast_result = model.predict(future)
        forecast = forecast_result["yhat"].tail(
            FORECAST_DAYS
        ).values
    future_dates = pd.date_range(
        start=df.index[-1] + pd.Timedelta(days=1),
        periods=FORECAST_DAYS
    )

    forecast_df = pd.DataFrame({

        "Date": future_dates,

        "Predicted_Sales": forecast
    })

    console.print(
        Panel(
            str(forecast_df),
            title="Next 7 Days Forecast"
        )
    )

    return {
        "forecast_df": forecast_df
    }

In [19]:
# =========================================================
# BUILD GRAPH
# =========================================================

builder = StateGraph(ForecastState)

builder.add_node(
    "Planner",
    planner_agent
)

builder.add_node(
    "Executor",
    executor_agent
)

builder.add_node(
    "Checker",
    checker_agent
)

builder.add_node(
    "Decision",
    decision_agent
)

builder.add_node(
    "Tuning",
    tuning_agent
)

builder.add_node(
    "Forecast",
    forecasting_agent
)



In [20]:
# =========================================================
# FLOW
# =========================================================

builder.set_entry_point("Planner")

builder.add_edge(
    "Planner",
    "Executor"
)

builder.add_edge(
    "Executor",
    "Checker"
)

builder.add_edge(
    "Checker",
    "Decision"
)

builder.add_edge(
    "Decision",
    "Tuning"
)

builder.add_edge(
    "Tuning",
    "Forecast"
)

builder.add_edge(
    "Forecast",
    END
)



In [21]:
# =========================================================
# COMPILE GRAPH
# =========================================================

graph = builder.compile()


# =========================================================
# VISUALIZE GRAPH
# =========================================================

try:

    print(
        graph.get_graph().draw_ascii()
    )

except:
    pass

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
 +---------+   
 | Planner |   
 +---------+   
      *        
      *        
      *        
+----------+   
| Executor |   
+----------+   
      *        
      *        
      *        
 +---------+   
 | Checker |   
 +---------+   
      *        
      *        
      *        
+----------+   
| Decision |   
+----------+   
      *        
      *        
      *        
  +--------+   
  | Tuning |   
  +--------+   
      *        
      *        
      *        
+----------+   
| Forecast |   
+----------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [22]:
# =========================================================
# RUN GRAPH
# =========================================================

final_state = graph.invoke({

    "models": [],

    "results": [],

    "best_model": "",

    "tuning_method": "",

    "best_params": {},

    "forecast_df": None,

    "error": None
})

Planner Agent Running

Selected Models → ['ARIMA', 'SARIMAX', 'Prophet']

Executor Agent Running

ARIMA Completed

SARIMAX Completed

...

Best Tuning Method → GridSearch

Best Params → {'d': 1, 'p': 1, 'q': 2}

Forecasting Agent

2026-07-13 10:11:38,902 - INFO - Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
2026-07-13 10:11:38,902 - INFO - Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
2026-07-13 10:11:38,906 - INFO - n_changepoints greater than number of observations. Using 17.


╭───────────────────────────────────────────── Next 7 Days Forecast ──────────────────────────────────────────────╮
│         Date  Predicted_Sales                                                                                   │
│ 0 2024-01-31        70.565730                                                                                   │
│ 1 2024-02-01        74.236068                                                                                   │
│ 2 2024-02-02        77.727238                                                                                   │
│ 3 2024-02-03        71.728111                                                                                   │
│ 4 2024-02-04        77.568043                                                                                   │
│ 5 2024-02-05        76.753352                                                                                   │
│ 6 2024-02-06        77.252589                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [23]:
# =========================================================
# FINAL FORECAST OUTPUT
# =========================================================

forecast_df = final_state["forecast_df"]

forecast_df.to_csv(
    "Forecast_Output.csv",
    index=False
)

console.print(
    "\n[bold green]"
    "Forecast Saved → Forecast_Output.csv"
    "[/bold green]"
)

Forecast Saved → Forecast_Output.csv